<a href="https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/tuning/role_kinds_tuning.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

# The Interpretability Role Kinds in One Phased Run

Fine-Tuning Scheduler's tunable views give every tensor a role (venngram's `venngram.roles`), and a schedule selects
tensors by role. `base` and `adapter` are the familiar ones. This notebook shows the other four, each doing what it is
for, in one small model and one schedule:

| Role | What it is for | Thawed by a phase | Optimized | Checkpointed |
|---|---|---|---|---|
| `artifact` | a published or saved tensor applied in every forward pass, reloaded from its source | never | never | no |
| `latent:<name>` | a latent model (an SAE, a transcoder) the run tunes | when a phase names it | yes | yes |
| `intervention:<name>` | a trainable intervention at a site, here a steering vector | when a phase names it | yes | yes |
| `fitted` | state computed during the run without gradients, here a mean-difference direction | never | never | yes |

and a **promotion**: a later phase turns the `artifact` into `latent:sae` and fine-tunes it, which is how a frozen
analysis artifact becomes a trained component mid-run. (Earlier releases called `artifact` `instrument`.)

Every role is selected by address (`role:...`), never by parameter name. The run is GPT-2 on RTE through
Interpretune's core loop, small enough for a CPU.

Requirements: `uv pip install -e ".[examples]"` (CPU torch is fine).

> **Until the `speediedan/rte` component is public (#261), this notebook needs a Hugging Face token with access to
> it** (set `HF_TOKEN`). No GPU and no license acceptance are needed.

In [2]:
# Parameters - These will be injected by papermill during parameterized test runs
LIMIT_TRAIN_BATCHES = 8  # batches per epoch
LIMIT_VAL_BATCHES = 4
FIT_BATCHES = 8  # training batches the fitted direction is computed from
SEED = 0
REVISION = (
    None  # component snapshot; None resolves the recorded pin (pass a local-publish rev to validate unmerged configs)
)

In [4]:
# @title Imports { display-mode: "form" }
import os
import tempfile
from collections import Counter
from pathlib import Path

import torch
from finetuning_scheduler.core.checkpoint import load_payload
from torch import nn

import interpretune as it
from interpretune.config import SessionRunnerCfg
from interpretune.runners.core import SessionRunner
from interpretune.tuning.core_loop import FTSCoreCallback
from interpretune.tuning.interventions import OpIntervention, attach_intervention
from interpretune.tuning.latents import LatentSite, attach_latent_model
from it_examples.seeds import ensure_local_seeds
from it_examples.utils.nb_ui_utils import display_notebook_table

torch.manual_seed(SEED)

## 1. One Model, Four Role Kinds

The session is the `rte_ft.gpt2.core` configuration of the `speediedan/rte` component: GPT-2 answering RTE through
Interpretune's core loop. A module attaches what it tunes in `register_tunable_modules()`, which runs once the model
exists and before its view is built. This one attaches three things:

- **`sae`**, a published sparse autoencoder (OpenAI's 32k-latent residual SAE for GPT-2 small's layer 1, trained on
  GPT-2's own weights, loaded through sae_lens), spliced in at `point:hook_resid_post & layers:1` with the `replace`
  op, so the model runs on the SAE's reconstruction. It is attached `frozen=True`, so its tensors are `artifact`.
- **`steer`**, a steering vector (`OpIntervention("add", 768)`) at `point:hook_resid_post & layers:8`. Its operand starts
  at zero, where `add` is the identity, and its tensor is `intervention:steer`.
- **`mean_diff`**, a mean-difference direction at `point:hook_resid_post & layers:4`, held in a buffer. It is attached
  with `frozen="fitted"`: computed in the run, saved with it, never trained.

The sites are venngram addresses, so the same lines place them on any composition whose architecture Interpretune maps.

In [5]:
# @title 1: The session and what it attaches { display-mode: "form" }
D_MODEL = 768
SAE_RELEASE, SAE_ID = "gpt2-small-resid-post-v5-32k", "blocks.1.hook_resid_post"


class FittedDirection(nn.Module):
    """A direction held in a buffer: an operand the run computes and saves, never one it trains."""

    def __init__(self, d: int) -> None:
        super().__init__()
        self.register_buffer("direction", torch.zeros(d))

    def forward(self) -> torch.Tensor:
        return self.direction


def load_sae() -> nn.Module:
    from sae_lens import SAE

    loaded = SAE.from_pretrained(SAE_RELEASE, SAE_ID, device="cpu")
    return loaded[0] if isinstance(loaded, tuple) else loaded


ensure_local_seeds()  # idempotent, offline: seed publish sources -> local components cache
if REVISION is None:  # a validation run passes a local-publish REVISION and needs none of these steps
    it.hub.pull("speediedan/rte")  # caches the component's manifest and entrypoint; executes nothing
    os.environ["IT_TRUST_REMOTE_CODE"] = "1"  # opt in to running its entrypoint, for this session only
    it.hub.pull("speediedan/rte", "rte_ft.gpt2.core")
base_itdm_cfg, base_it_cfg, dm_cls, m_cls = it.hub.load("speediedan/rte", "rte_ft.gpt2.core", revision=REVISION)


class RoleKindsModule(m_cls):
    def register_tunable_modules(self) -> None:
        super().register_tunable_modules()
        attach_latent_model(
            self, "sae", load_sae(), LatentSite("point:hook_resid_post & layers:1", op="replace"), frozen=True
        )
        attach_intervention(self, "steer", OpIntervention("add", D_MODEL), "point:hook_resid_post & layers:8")
        attach_intervention(
            self,
            "mean_diff",
            OpIntervention("add", FittedDirection(D_MODEL)),
            "point:hook_resid_post & layers:4",
            frozen="fitted",
        )


it_session = it.ITSession(
    it.ITSessionConfig(
        adapter_ctx=(it.Adapter.core,),
        datamodule_cfg=base_itdm_cfg,
        module_cfg=base_it_cfg,
        datamodule_cls=dm_cls,
        module_cls=RoleKindsModule,
    )
)
it.it_init(**it_session)
module = it_session.module
roles = module.tunable_view.roles()
display_notebook_table(
    [[name, f"`{role}`"] for name, role in roles.items() if role != "base"],
    ["Tensor", "Role"],
    title="The roles the module's view declares (every other tensor is `base`)",
    caption=f"Role counts over all {sum(1 for _ in module.named_parameters())} parameters and the fitted buffer: "
    + ", ".join(f"{role} {n}" for role, n in sorted(Counter(roles.values()).items())),
)

src/interpretune/hub/components.py:568: 'speediedan/rte' resolved to the local-publish snapshot locala15bee8, which shadows the cached Hub revision(s) ['14d388e1b6a2', '22152f0e349b', '4db2f48ce89f', '67fff0a10dd0', '738e41229e39', '7d0bbe4d887c', '848038b1310a', '8f61b54c4d2c', 'a9b2407a96e5', 'b10c7651eb32']; the Hub was not consulted (resolution is cache-only). If you meant the published artifact, run interpretune.hub.pull('speediedan/rte') or resolve with require_hub=True.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'GPT2Config' object has no attribute 'quantization_config'


[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'GPT2Config' object has no attribute '_pre_quantization_dtype'


[INFO] interpretune.utils.logging: Preparing data: InterpretunableDataModule


[INFO] interpretune.utils.logging: The following columns  don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: hypothesis, idx, label, premise, sequences. If hypothesis, idx, label, premise, sequences are not expected by `GPT2LMHeadModel.forward`, you can safely ignore this message.


... (5 outputs left out)

[INFO] interpretune.utils.logging: Setting up datamodule: InterpretunableDataModule


[INFO] interpretune.utils.logging: Setting up model: InterpretunableModule


[INFO] interpretune.utils.logging: initializing optimizers and schedulers: InterpretunableModule


Tensor,Role
latent_models.sae.b_dec,`artifact`
latent_models.sae.W_dec,`artifact`
latent_models.sae.W_enc,`artifact`
latent_models.sae.b_enc,`artifact`
interventions.steer.operand.value,`intervention:steer`
interventions.mean_diff.operand.direction,`fitted`


## 2. The Schedule

Two phases, each a stage of the experiment, every entry an address:

```python
{
    0: {"params": ["role:intervention:steer"], "max_transition_epoch": 1},
    1: {
        "params": ["role:latent:sae"],
        "promote": {"role:artifact": "latent:sae"},
        "refreeze": ["role:intervention:steer"],
    },
}
```

Phase 0 trains the steering vector alone. Phase 1 **promotes** the SAE's `artifact` tensors to `latent:sae`, thaws
them, and holds the steering vector where phase 0 left it (`refreeze`). The base model is never named, so it never
trains; `fitted` and `artifact` cannot be thawed by any phase. `promote` and `refreeze` run on the core loop
(`FTSCoreCallback`); the Lightning callback refuses them.

**The fitted direction** is computed by a phase hook at the start of phase 0, before any gradient step: the mean
residual at layer 4, at the answer position, of the entailed examples minus that of the others, over a few training
batches. It is computed with the intervention still at zero (the identity) and written into its buffer under
`torch.no_grad()`.

A small core-loop callback records each role group at the end of each phase's epoch: whether it could take a
gradient, whether the phase's optimizer held it, and its values. It runs before the tuning scheduler's own callback,
so it reads each phase before the next one starts. (A phase's start hooks run before its `promote` and its thaw, so
they see the previous phase's state.)

In [6]:
# @title 2: Fit, record, and run { display-mode: "form" }
GROUPS = {
    "artifact / latent:sae": "latent_models.sae.",
    "intervention:steer": "interventions.steer.",
    "fitted": "interventions.mean_diff.",
}
records: dict[int, dict] = {}


def group_of(name: str) -> str | None:
    return next((g for g, prefix in GROUPS.items() if name.startswith(prefix)), None)


@torch.no_grad()
def fit_mean_diff(event) -> None:
    """Start hook, phase 0: the mean answer-position residual after block 4, entailed minus not entailed."""
    if event.depth != 0:
        return
    hf_model = module.model
    sums, counts = torch.zeros(2, D_MODEL), torch.zeros(2)
    for i, batch in enumerate(module.datamodule.train_dataloader()):
        if i >= FIT_BATCHES:
            break
        out = hf_model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], output_hidden_states=True)
        resid = out.hidden_states[5][:, -1].float()  # after block 4, at the answer position (left padding)
        labels = batch["labels"].long().view(-1)
        for cls in (0, 1):
            sums[cls] += resid[labels == cls].sum(0)
            counts[cls] += (labels == cls).sum()
    direction = sums[0] / counts[0].clamp(min=1) - sums[1] / counts[1].clamp(min=1)
    module.interventions.mean_diff.operand.direction.copy_(direction)


def snapshot(optimizer=None) -> dict:
    """Each role group as it stands: whether any tensor can take a gradient, whether the optimizer holds it, and a
    copy of its values."""
    optimized = {id(p) for g in optimizer.param_groups for p in g["params"]} if optimizer is not None else set()
    record = {}
    for name, p in [*module.named_parameters(), *module.named_buffers()]:
        group = group_of(name)
        if group is not None:
            entry = record.setdefault(group, {"thawed": False, "optimized": False, "tensors": {}})
            entry["thawed"] |= bool(getattr(p, "requires_grad", False))
            entry["optimized"] |= id(p) in optimized
            entry["tensors"][name] = p.detach().clone()
    return record


class RecordEpochs:
    """A core-loop callback recording each epoch as it ends, before the tuning scheduler moves to its next phase."""

    def on_train_epoch_end(self, ctx, metrics) -> None:
        records[fts.tuning_scheduler.curr_depth] = snapshot(ctx.optimizer)


initial = snapshot()
checkpoint_dir = Path(tempfile.mkdtemp())
schedule = {
    0: {"params": ["role:intervention:steer"], "max_transition_epoch": 1},
    1: {
        "params": ["role:latent:sae"],
        "promote": {"role:artifact": "latent:sae"},
        "refreeze": ["role:intervention:steer"],
    },
}
fts = FTSCoreCallback(
    schedule,
    tuning_scheduler_kwargs={
        "checkpoint": {
            "class_path": "finetuning_scheduler.core.checkpoint.BestCheckpoint",
            "init_args": {"dirpath": str(checkpoint_dir), "monitor": "val_loss", "save_initial": True},
        },
        "phase_hooks": {"start": [fit_mean_diff]},
    },
)
SessionRunner(
    run_cfg=SessionRunnerCfg(
        it_session=it_session,
        max_epochs=2,
        limit_train_batches=LIMIT_TRAIN_BATCHES,
        limit_val_batches=LIMIT_VAL_BATCHES,
        callbacks=[RecordEpochs(), fts],  # the recorder first, so it reads each phase before a transition
    )
).train()
tuning_scheduler = fts.tuning_scheduler
assert tuning_scheduler.curr_depth == 1, "both phases must run"
print("fitted direction norm:", f"{module.interventions.mean_diff.operand.direction.norm():.2f}")

[INFO] interpretune.utils.logging: Preparing data: InterpretunableDataModule


[INFO] interpretune.utils.logging: The following columns  don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: hypothesis, idx, label, premise, sequences. If hypothesis, idx, label, premise, sequences are not expected by `GPT2LMHeadModel.forward`, you can safely ignore this message.


[INFO] interpretune.utils.logging: The following columns  don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: hypothesis, idx, label, premise, sequences. If hypothesis, idx, label, premise, sequences are not expected by `GPT2LMHeadModel.forward`, you can safely ignore this message.


[INFO] interpretune.utils.logging: The following columns  don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: hypothesis, idx, label, premise, sequences. If hypothesis, idx, label, premise, sequences are not expected by `GPT2LMHeadModel.forward`, you can safely ignore this message.


Saving the dataset (0/1 shards):   0%|          | 0/2490 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/277 [00:00<?, ? examples/s]

... (6 outputs left out)


Epoch 1 end: {'train_loss': 11.6372, 'val_loss': 8.1304, 'accuracy': 0.4375}


[INFO] interpretune.utils.logging: Running stage end hooks on IT module: InterpretunableModule


[INFO] interpretune.utils.logging: Running stage end hooks on IT datamodule: InterpretunableDataModule


fitted direction norm: 12.00


## 3. What Each Role Guaranteed

The first table is read at the end of each phase: whether any of a group's tensors could take a gradient (`thawed`),
whether the phase's optimizer held them (`optimized`), and whether their values changed during the phase. The second
reads every checkpoint the run saved and lists which groups it contains.


In [7]:
# @title 3: The guarantees, as observed { display-mode: "form" }
before = {0: initial, 1: records[0]}


def changed(group: str, phase: int) -> bool:
    return any(
        not torch.equal(records[phase][group]["tensors"][name], value)
        for name, value in before[phase][group]["tensors"].items()
    )


yes = {True: "yes", False: "no"}
display_notebook_table(
    [
        [
            group,
            phase,
            yes[records[phase][group]["thawed"]],
            yes[records[phase][group]["optimized"]],
            yes[changed(group, phase)],
        ]
        for group in GROUPS
        for phase in sorted(records)
    ],
    ["Role group", "Phase", "Thawed", "Optimized", "Changed in the phase"],
    title="Each role group during each phase",
)
sae, steer, fitted = GROUPS
assert not records[0][sae]["thawed"] and not changed(sae, 0), (
    "the artifact is applied but never trained before its promotion"
)
assert records[1][sae]["optimized"] and changed(sae, 1), "promoted, it trains"
assert records[0][steer]["optimized"] and changed(steer, 0), "the steering vector trains in phase 0"
assert not records[1][steer]["thawed"] and not changed(steer, 1), "refrozen, it holds where phase 0 ended"
assert not any(records[d][fitted]["thawed"] or records[d][fitted]["optimized"] for d in records)
assert not changed(fitted, 1), "the fitted direction holds once computed (in phase 0, by the start hook)"
assert any(v.abs().sum() > 0 for v in records[1][fitted]["tensors"].values()), "the fitted direction was computed"

saved = []
for path in sorted(checkpoint_dir.glob("*.pt"), key=lambda p: p.stat().st_mtime):
    keys = load_payload(path)["model"].keys()
    groups = {group_of(k) for k in keys} - {None}
    saved.append([path.name, *("yes" if g in groups else "no" for g in GROUPS)])
display_notebook_table(
    saved,
    ["Checkpoint", *GROUPS],
    title="What each saved checkpoint holds, in the order the run wrote them",
    caption="Base tensors are in every checkpoint and are left out of the table.",
)

Role group,Phase,Thawed,Optimized,Changed in the phase
artifact / latent:sae,0,no,no,no
artifact / latent:sae,1,yes,yes,yes
intervention:steer,0,yes,yes,yes
intervention:steer,1,no,no,no
fitted,0,no,no,yes
fitted,1,no,no,no


Checkpoint,artifact / latent:sae,intervention:steer,fitted
best-1-0598233c.pt,no,yes,yes
best-2-a236fb83.pt,yes,yes,yes


**Reading the tables.**

- **`artifact`** is applied in every forward pass from the first step (the model runs on the SAE's reconstruction
  throughout) but is not thawed, not optimized and not in any checkpoint saved before its promotion: a run resumed from
  one of those reloads it from its source.
- **Promotion** changes all three at once. From phase 1 the same tensors are `latent:sae`: thawed, in the optimizer,
  and in the checkpoints saved after the promotion, because from then on the run owns their values.
- **`intervention:steer`** trains in phase 0 and is held fixed in phase 1 by `refreeze`, from where phase 0 ended. It
  is checkpointed throughout, because its value is the run's.
- **`fitted`** is never thawed or optimized. It changes once, in phase 0, when the start hook computes it (the table's
  "changed" for phase 0), and is in every checkpoint, so a resumed run keeps the direction the run computed rather
  than recomputing it.

Each phase's tensors were named by role alone: no schedule entry names a parameter.